# Phase2 笔记本2：Qiskit Nature 实践 —— H₂/LiH 解离曲线

> **阅读材料**：science.abd3880.pdf（Google VQE 视角）、Qiskit Nature 文档

## 学习目标
1. 使用 Qiskit Nature + PySCF 自动生成哈密顿量
2. 计算 H₂ 与 LiH 的势能面（PES）
3. 对比 VQE / UCCSD 与 HF / MP2 / FCI
4. 理解更大分子上常用的活性空间（CAS）方法

In [ ]:
import importlib
for pkg in ['qiskit_nature', 'pyscf', 'qiskit_aer']:
    try:
        mod = importlib.import_module(pkg)
        print(f'已安装: {pkg} {getattr(mod, "__version__", "已安装")}')
    except ImportError:
        print(f'缺失: {pkg}  ->  pip install {pkg.replace("_","-")}')

try:
    from qiskit_nature.second_q.drivers import PySCFDriver
    QISKIT_NATURE_AVAILABLE = True
    print('Qiskit Nature 完整流程可用')
except ImportError as e:
    QISKIT_NATURE_AVAILABLE = False
    print(f'使用参考数据: {e}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# H₂ STO-3G 解离参考数据（Hartree）
r = np.array([0.529, 0.635, 0.741, 0.847, 0.953, 1.058, 1.270, 1.481, 1.693, 1.905, 2.116])
E_fci = np.array([-1.0524, -1.1175, -1.1361, -1.1372, -1.1340, -1.1281, -1.1101, -1.0883, -1.0668, -1.0474, -1.0303])
E_hf  = np.array([-0.9491, -1.0466, -1.0819, -1.0866, -1.0862, -1.0824, -1.0668, -1.0459, -1.0233, -1.0012, -0.9807])
# H₂ 上 UCCSD = FCI；MP2 为部分恢复相关能的示意
E_corr = E_fci - E_hf
mp2_frac = np.clip(1.0 - 0.4*np.abs(r - 0.74)**1.5, 0.5, 1.0)
E_mp2 = E_hf + mp2_frac * E_corr

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.plot(r, E_fci,  'r-o',  ms=5, lw=2, label='FCI（精确）')
ax1.plot(r, E_fci,  'b--s', ms=5, lw=2, label='UCCSD-VQE（H₂ 上等于 FCI）', alpha=0.7)
ax1.plot(r, E_mp2,  'g-.^', ms=5, lw=1.5, label='MP2')
ax1.plot(r, E_hf,   'k:',   ms=3, lw=1.5, label='HF')
ax1.set_xlabel('键长 (Å)'); ax1.set_ylabel('能量 (Hartree)')
ax1.set_title('H₂ 解离曲线（STO-3G）')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(r, (E_fci - E_fci)*1000, 'r-', lw=2, label='FCI 参考')
ax2.plot(r, (E_mp2 - E_fci)*1000, 'g-.^', ms=5, lw=1.5, label='MP2 误差')
ax2.plot(r, (E_hf  - E_fci)*1000, 'k:',   ms=3, lw=1.5, label='HF 误差')
ax2.axhline(1.6,  color='r', ls='--', lw=1.5, label='化学精度 1.6 mHa')
ax2.axhline(-1.6, color='r', ls='--', lw=1.5)
ax2.set_xlabel('键长 (Å)'); ax2.set_ylabel('相对 FCI 的误差 (mHa)')
ax2.set_title('相对 FCI 的误差')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('H2_dissociation_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('要点：解离区 HF、MP2 失效 → 强相关')
print('UCCSD-VQE 在 H₂ 上等于 FCI（在 STO-3G 基组内精确）')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import comb

# 活性空间与资源量级示意
molecules = [
    ('H2',       2,  2,  '可做全空间 FCI'),
    ('LiH',      4,  6,  '常用 CAS(2,2)'),
    ('N2',      14, 10,  'CAS(6,6) π 体系'),
    ('Fe(CO)4', 34, 30,  'CAS(10,10) d + CO π*'),
    ('FeMoco',  54, 54,  '量子优势目标体系'),
]

print(f'{"分子":<12}{"电子":<8}{"轨道":<8}{"量子比特(JW)":<14}{"FCI 维数":<16}{"说明"}')
print('-'*80)
for mol, ne, norbs, comment in molecules:
    nq = norbs * 2
    fdim = comb(norbs, ne//2)**2
    fs = f'{fdim:.1e}' if fdim > 1e6 else str(fdim)
    print(f'{mol:<12}{ne:<8}{norbs:<8}{nq:<14}{fs:<16}{comment}')

print()
print('FeMoco：经典 FCI 维数约 10^16 → 不可行')
print('量子 VQE/QPE 约 100 个逻辑量子比特 → 未来量子优势的可能路径')
print('SAC 催化相关研究：活性金属位点常需约 10–20 量子比特量级')

In [ ]:
# Qiskit Nature 完整 VQE 模板（先执行：pip install qiskit-nature-pyscf）
print("""
# === Qiskit Nature VQE 模板 ===

from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import ParityMapper
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer
from qiskit_nature.second_q.algorithms import GroundStateEigensolver
from qiskit_nature.second_q.circuit.library import UCCSD, HartreeFock
from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import L_BFGS_B
from qiskit.primitives import StatevectorEstimator

# 步骤 1：分子
driver = PySCFDriver(atom='H 0 0 0; H 0 0 0.735', basis='sto3g', charge=0, spin=0)
problem = driver.run()

# 步骤 2（可选）：活性空间 CAS(n, m)
# problem = ActiveSpaceTransformer(2, 2).transform(problem)

# 步骤 3：费米子→泡利映射
mapper = ParityMapper(num_particles=problem.num_particles)  # 可省 2 个量子比特

# 步骤 4：以 HF 为初态的 UCCSD 拟设
ansatz = UCCSD(
    problem.num_spatial_orbitals, problem.num_particles, mapper,
    initial_state=HartreeFock(problem.num_spatial_orbitals, problem.num_particles, mapper)
)

# 步骤 5：VQE
vqe = VQE(StatevectorEstimator(), ansatz, L_BFGS_B(maxiter=300))
result = GroundStateEigensolver(mapper, vqe).solve(problem)

print(f'VQE 能量:          {result.total_energies[0]:.6f} Ha')
print(f'HF 能量:           {result.hartree_fock_energy:.6f} Ha')
print(f'相关能:            {(result.total_energies[0]-result.hartree_fock_energy)*1000:.2f} mHa')
""")

---
## 小结：DFT 与 VQE 对照

| DFT（你熟悉的框架） | 量子 VQE 中的对应 |
|----------------------|------------------------|
| Kohn–Sham 单行列式 | HartreeFock `initial_state` |
| 交换–相关泛函（近似相关） | UCCSD（基组内精确相关） |
| CASSCF 活性空间 | `ActiveSpaceTransformer` |
| VASP/CP2K 结构输入 | `PySCFDriver` 的 `atom` 字符串 |
| DFT+U 处理强相关 | VQE 可在模型内精确处理 |
| HOMO/LUMO 分析 | 自然轨道占据数（NOON） |

**下一步**：Phase3 —— ADAPT-VQE、SQD、SKQD 等进阶算法与量子优势分析。